In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação por variáveis + Park (comparação) com blocos RAW/ALIGNED e RF-Temp reforçado.

Pipeline resumido:
1) Carrega e recorta 30–50 kHz, encontra colunas comuns, define y_ref (= mediana real @REF_TEMP).
2) Extrai features globais interpretáveis (mean, std, amp, slope, peak_pos_rel, centroid, skew, kurtosis, energias em 3 bandas).
3) RF-Temp: varre um grid pequeno e faz K-fold (quando possível); senão usa fallback estável.
4) Compensação por variáveis (offset + gain + tilt) + micro-shift para aproximar centroid (Hz) e âncora nos extremos (curve hugs the reference baseline).
5) Métricas em 4 blocos: (Grupo 1) vs REF RAW/ALIGNED e (Grupo 2) vs ORIG RAW/ALIGNED.
6) Park (1999) fiel ao TCC — só COMPARAÇÃO — com plot separado.
7) Plots: um para RF e outro para Park.
"""

import re, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

# ===================== PARÂMETROS =====================
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"

# Conjuntos fixos
TEMPS_TREINO = {0, 10, 40, 60}
TEMPS_PROVA  = {-10, 30, 50, 70}

# Banda e compensação
SMOOTH_WIN          = 7    # suavização final; ímpar => ativa
TAU_MAX_FRAC        = 0.02 # micro-shift máx para aproximar centroid (fração da largura em Hz)
ANCHOR_TO_REF_ENDS  = True # ancora extremos da curva ao da REF

# RF-Temp (grid compacto + fallback)
RF_GRID = [
    dict(n_estimators=600, max_depth=18, min_samples_leaf=2, min_samples_split=4, max_features="sqrt", bootstrap=True, n_jobs=-1, random_state=17),
    dict(n_estimators=800, max_depth=22, min_samples_leaf=2, min_samples_split=4, max_features="sqrt", bootstrap=True, n_jobs=-1, random_state=7),
    dict(n_estimators=1000, max_depth=None, min_samples_leaf=1, min_samples_split=4, max_features="sqrt", bootstrap=True, n_jobs=-1, random_state=42),
]

# Park (comparação, fora do RF)
PARK_MAX_SHIFT_FRAC = 0.25 # até 25% do nº de pontos
PARK_OVERLAP_MIN    = 0.60 # fração mínima de overlap
PARK_SMOOTH_WIN     = 5

# Caps de segurança na compensação por variáveis
CAP_GAIN_FRAC   = 0.60
CAP_OFFSET_FRAC = 0.60
CAP_TILT_FRAC   = 0.60

# ===================== HELPERS BÁSICOS =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def moving_average(arr, win):
    if win<=1 or win%2==0: return arr
    r=win//2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s/float(win)

def shift_interp(x_row, fhz, tau_hz):
    """Desloca no eixo de frequência por interp1d simples (np.interp)."""
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

# ===================== MÉTRICAS =====================
def rmsd(y_ref, y):   return float(np.sqrt(np.mean((y_ref - y)**2)))

def ccdm(y_ref, y):
    y1, y2 = y_ref - y_ref.mean(), y - y.mean()
    den = (np.linalg.norm(y1)*np.linalg.norm(y2)) + 1e-12
    rho = float(np.clip(np.dot(y1, y2)/den, -1, 1))
    return 1.0 - rho

def corr_per_sample(Y, Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        num=((y-y.mean())*(yh-yh.mean())).sum()
        den=np.sqrt(((y-y.mean())**2).sum()*((yh-yh.mean())**2).sum())+1e-12
        out.append(float(np.clip(num/den,-1,1)))
    return np.array(out)

def sam_per_sample(Y,Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        den=(np.linalg.norm(y)*np.linalg.norm(yh))+1e-12
        cosang=float(np.clip(np.dot(y,yh)/den,-1,1))
        out.append(float(np.degrees(np.arccos(cosang))))
    return np.array(out)

def nrmse_per_sample(Y,Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        rmse=np.sqrt(np.mean((y-yh)**2))
        rng=np.max(y)-np.min(y)
        out.append(float(rmse/(rng+1e-12)))
    return np.array(out)

def eval_all_metrics(Y_true, Y_pred):
    y1, y2 = Y_true.reshape(-1), Y_pred.reshape(-1)
    Corr = float(corr_per_sample(Y_true, Y_pred).mean())
    return dict(
        R2      = r2_score(y1, y2),
        RMSE    = float(np.sqrt(mean_squared_error(y1, y2))),
        MAE     = float(mean_absolute_error(y1, y2)),
        Corr    = Corr,
        SAM_deg = float(sam_per_sample(Y_true, Y_pred).mean()),
        NRMSE   = float(nrmse_per_sample(Y_true, Y_pred).mean()),
        RMSD    = float(np.mean([rmsd(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
        CCDM    = float(np.mean([ccdm(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
    )

def print_block(title, m):
    print(f"\n== {title} ==")
    print(" | ".join([f"{k}={m[k]:.4f}" for k in ["R2","RMSE","MAE","Corr","SAM_deg","NRMSE","RMSD","CCDM"]]))

# --------- features auxiliares para métricas dos grupos
def energy_weighted_centroid(f, x):
    xm = np.asarray(x, float)
    w  = xm*xm
    den = float(np.trapezoid(w, f))
    if den <= 1e-18: return float(np.mean(f))
    num = float(np.trapezoid(f*w, f))
    return num/den

def slope_over_band(f, x):
    return float((x[-1]-x[0])/(f[-1]-f[0] + 1e-12))

def group_metrics(f, A, B):
    # métricas “básicas”
    Corr   = float(corr_per_sample(A[None,:], B[None,:]).mean())
    SAMdeg = float(sam_per_sample(A[None,:], B[None,:]).mean())
    CCDM_  = float(ccdm(A, B))
    RMSD_  = float(rmsd(A, B))
    # diferenças de atributos globais
    mean_diff = float(np.mean(B) - np.mean(A))
    amp_diff  = float((B.max()-B.min()) - (A.max()-A.min()))
    centA = energy_weighted_centroid(f, A); centB = energy_weighted_centroid(f, B)
    centroid_diff_hz = float(centB - centA)
    slope_diff = float(slope_over_band(f, B) - slope_over_band(f, A))
    return dict(
        Corr=Corr, SAM_deg=SAMdeg, CCDM=CCDM_, RMSD=RMSD_,
        mean_diff=mean_diff, amp_diff=amp_diff,
        centroid_diff_hz=centroid_diff_hz, slope_diff=slope_diff
    )

# ===== overlay/align por correlação + LS (só para AVALIAR, não altera o pipeline)
def best_affine_overlay(f, ref, sig, max_lag_bins=None):
    ref = np.asarray(ref, float); sig = np.asarray(sig, float)
    n = len(ref)
    if max_lag_bins is None:
        max_lag_bins = max(1, int(0.2*n))
    # z-score (evita viés)
    rz = (ref - ref.mean())/(ref.std()+1e-12)
    sz = (sig - sig.mean())/(sig.std()+1e-12)
    # correlação cruzada
    lags = np.arange(-max_lag_bins, max_lag_bins+1)
    best = ( -np.inf, 0 )
    for lag in lags:
        if   lag > 0: r = rz[lag:]; s = sz[:n-lag]
        elif lag < 0: r = rz[:n+lag]; s = sz[-lag:]
        else:         r = rz; s = sz
        if len(r) < max(10, int(0.2*n)): continue
        cc = float(np.dot(r, s)/max(1, len(r)))
        if cc > best[0]: best = (cc, lag)
    best_lag = int(best[1])
    # reconstrói o sinal deslocado no domínio ORIGINAL (não z-score)
    if   best_lag > 0: s_use = sig[:n-best_lag]; r_use = ref[best_lag:]
    elif best_lag < 0: s_use = sig[-best_lag:];  r_use = ref[:n+best_lag]
    else:              s_use = sig;             r_use = ref
    # LS: r_use ~ a*s_use + b
    Su  = np.column_stack([s_use, np.ones_like(s_use)])
    sol = np.linalg.lstsq(Su, r_use, rcond=None)[0]
    a, b = float(sol[0]), float(sol[1])
    # aplica ao vetor completo (fazendo “shift por índice” e preenchendo bordas)
    out = np.empty_like(sig); out[:] = np.nan
    if   best_lag > 0:
        out[best_lag:] = a*sig[:n-best_lag] + b
        out[:best_lag] = (a*sig[0] + b)
    elif best_lag < 0:
        out[:n+best_lag] = a*sig[-best_lag:] + b
        out[n+best_lag:] = (a*sig[-1] + b)
    else:
        out = a*sig + b
    return best_lag, a, b, out

# ===================== FEATURES PARA RF-TEMP =====================
def compute_features(X, f):
    """
    Pacote de features globais:
      mean, std, amp, slope, peak_pos_rel, centroid (energia x^2),
      skew, kurtosis, energias E_low/E_mid/E_high por terços de banda.
    """
    X = np.asarray(X, float); n, m = X.shape
    out=[]
    # bandas (3 terços)
    cuts = np.linspace(f[0], f[-1], 4)
    for i in range(n):
        x = X[i]
        mean  = float(np.mean(x))
        std   = float(np.std(x))
        amp   = float(x.max() - x.min())
        slope = slope_over_band(f, x)
        pk_i  = int(np.argmax(x)); peak_pos_rel = pk_i / max(1,(m-1))
        # centroid energia
        centroid = energy_weighted_centroid(f, x)
        # skew/kurtosis no z-score
        z = (x - mean)/(std + 1e-12)
        skew = float(np.mean(z**3))
        kurt = float(np.mean(z**4))
        # energias por terço (trapezoid)
        mask_low  = (f>=cuts[0]) & (f<cuts[1])
        mask_mid  = (f>=cuts[1]) & (f<cuts[2])
        mask_high = (f>=cuts[2]) & (f<=cuts[3])
        def eband(mask):
            if mask.sum()<2: return 0.0
            return float(np.trapezoid((x[mask]**2), f[mask]))
        E_low  = eband(mask_low)
        E_mid  = eband(mask_mid)
        E_high = eband(mask_high)
        out.append([mean,std,amp,slope,peak_pos_rel,centroid,skew,kurt,E_low,E_mid,E_high])
    cols = ["mean","std","amp","slope","peak_pos_rel","centroid","skew","kurt","E_low","E_mid","E_high"]
    return np.array(out, float), cols

def fit_feature_vs_temp_models(F, T, names):
    models = {}
    T = np.asarray(T, float).reshape(-1,1)
    for j, name in enumerate(names):
        lr = LinearRegression().fit(T, F[:,j])
        models[name] = lr
    return models

def feature_targets_at_ref(models, ref_temp=REF_TEMP):
    Tref = np.array([[ref_temp]])
    return {name: float(lr.predict(Tref)[0]) for name,lr in models.items()}

# ===================== COMPENSAÇÃO POR VARIÁVEIS =====================
def apply_compensation_by_features(x, f, targets, caps, y_ref=None):
    """
    1) Offset -> média (mean)
    2) Gain   -> amplitude (amp)
    3) Tilt   -> slope
    4) Micro-shift de frequência para aproximar centroid
    5) Âncora nos extremos (opcional) para “voltar” à linha de referência
    """
    x = x.copy()
    mean_t = targets["mean"]
    amp_t  = targets["amp"]
    slope_t= targets["slope"]
    centroid_t = targets.get("centroid", None)

    mean_x = float(x.mean())
    amp_x  = float(x.max() - x.min())
    slope_x= slope_over_band(f, x)

    # 1) offset
    offset = mean_t - mean_x
    offset_cap = caps["offset_frac"] * max(1e-9, amp_x)
    offset = float(np.clip(offset, -offset_cap, offset_cap))
    x = x + offset

    # 2) ganho (em torno da média-alvo)
    gain = 1.0 if amp_x<=1e-9 else float(amp_t/amp_x)
    gmin = 1.0 - caps["gain_frac"]; gmax = 1.0 + caps["gain_frac"]
    gain = float(np.clip(gain, gmin, gmax))
    x = mean_t + gain*(x - mean_t)

    # 3) tilt (ajustar slope)
    delta_slope = slope_t - slope_x
    u = np.linspace(-0.5, 0.5, len(x))
    df = (f[-1]-f[0] + 1e-12)
    tilt_signal = (delta_slope * df) * u
    tilt_cap = caps["tilt_frac"] * max(1e-9, amp_x)
    tilt_signal = np.clip(tilt_signal, -tilt_cap, tilt_cap)
    x = x + tilt_signal

    # 4) micro-shift p/ aproximar centroid (suave)
    if centroid_t is not None:
        cent_x = energy_weighted_centroid(f, x)
        delta_c = centroid_t - cent_x
        tau_max = TAU_MAX_FRAC * (f[-1]-f[0])
        tau = float(np.clip(delta_c, -tau_max, tau_max))
        if abs(tau) > 1e-12:
            x = shift_interp(x, f, tau)

    # 5) âncora nos extremos vs REF (opcional)
    if ANCHOR_TO_REF_ENDS and (y_ref is not None):
        e0 = x[0]   - y_ref[0]
        e1 = x[-1]  - y_ref[-1]
        corr = np.linspace(e0, e1, len(x))
        x = x - corr

    return x

def compensate_set_by_features(X, f, feat_models, ref_temp, y_ref, caps, smooth_win=SMOOTH_WIN):
    targets = feature_targets_at_ref(feat_models, ref_temp)
    Y = np.zeros_like(X)
    for i in range(X.shape[0]):
        yi = apply_compensation_by_features(X[i], f, targets, caps, y_ref=y_ref)
        if smooth_win>1 and (smooth_win%2==1):
            yi = moving_average(yi, smooth_win)
        Y[i] = yi
    return Y, targets

# ===================== PARK (1999) – comparação =====================
def park_compensate_single(x, y_ref, max_shift_bins, overlap_min_frac=PARK_OVERLAP_MIN, smooth_win=PARK_SMOOTH_WIN):
    """
    Implementação empírica: varre deslocamento inteiro k em índices,
    ajusta δS = média(Y_ref - X_shift) no overlap e minimiza Va = soma((Y_ref - (X_shift+δS))^2).
    (Fiel ao espírito descrito no TCC.)
    """
    n = len(x)
    best = (np.inf, 0, 0.0)  # (Va, k, deltaS)
    for k in range(-max_shift_bins, max_shift_bins+1):
        if k < 0:
            xs = x[-k:n]
            yr = y_ref[0:n+k]
        elif k > 0:
            xs = x[0:n-k]
            yr = y_ref[k:n]
        else:
            xs = x
            yr = y_ref
        if len(xs) < int(overlap_min_frac*n):
            continue
        deltaS = float(np.mean(yr - xs))
        resid  = yr - (xs + deltaS)
        Va     = float(np.sum(resid*resid))
        if Va < best[0]:
            best = (Va, k, deltaS)
    _, kbest, dbest = best
    yout = np.empty_like(x); yout[:] = np.nan
    if kbest < 0:
        xs = x[-kbest:n] + dbest
        yout[0:n+kbest] = xs
        yout[n+kbest:]  = x[-1] + dbest
    elif kbest > 0:
        xs = x[0:n-kbest] + dbest
        yout[kbest:n] = xs
        yout[:kbest]  = x[0] + dbest
    else:
        yout = x + dbest
    if smooth_win>1 and smooth_win%2==1:
        yout = moving_average(yout, smooth_win)
    return yout, kbest, dbest

def park_batch(X, y_ref):
    n, m = X.shape
    max_shift_bins = max(1, int(PARK_MAX_SHIFT_FRAC * m))
    Y = np.zeros_like(X); lags=[]; deltas=[]
    for i in range(n):
        yi, k, d = park_compensate_single(X[i], y_ref, max_shift_bins)
        Y[i] = yi; lags.append(k); deltas.append(d)
    return Y, np.array(lags), np.array(deltas)

# ===================== CARGA =====================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

freq_cols_tr, _ = get_freq_columns(base_tr, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
freq_cols_te, _ = get_freq_columns(base_te, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
order = np.argsort(fhz); common_cols = [common_cols[i] for i in order]; fhz = fhz[order]
fkHz = fhz/1e3

X_tr_full = base_tr[common_cols].to_numpy(float)
X_te_full = base_te[common_cols].to_numpy(float)
T_tr_full = base_tr["temp_c"].to_numpy(float)
T_te_full = base_te["temp_c"].to_numpy(float)

pool_20=[]
if (base_tr["temp_c"]==REF_TEMP).any():
    pool_20.append(base_tr.loc[base_tr["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
if (base_te["temp_c"]==REF_TEMP).any():
    pool_20.append(base_te.loc[base_te["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
assert len(pool_20)>0, "Não há curva real @20°C!"
y_ref = np.median(np.vstack(pool_20), axis=0)

# restrições de treino/prova por temperaturas
tr_restr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
te_restr = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()

X_tr = tr_restr[common_cols].to_numpy(float)
X_te = te_restr[common_cols].to_numpy(float)
T_tr = tr_restr["temp_c"].to_numpy(float)
T_te = te_restr["temp_c"].to_numpy(float)

# ===================== RF-Temp REFORÇADO =====================
F_tr, feat_names = compute_features(X_tr, fhz)
F_te, _          = compute_features(X_te, fhz)

# seleção curta com K-fold (se possível); caso contrário, fallback
best_cv = -1e9; best_model=None; best_conf=None
n_samples = F_tr.shape[0]
if n_samples >= 6:
    k = min(5, n_samples//2)  # evitar folds com 1 amostra
    k = max(2, k)
    for conf in RF_GRID:
        rf = RandomForestRegressor(**conf)
        kf = KFold(n_splits=k, shuffle=True, random_state=123)
        cv_scores=[]
        for tr_idx, va_idx in kf.split(F_tr):
            Ftr, Fva = F_tr[tr_idx], F_tr[va_idx]
            Ttr, Tva = T_tr[tr_idx], T_tr[va_idx]
            if len(np.unique(Ttr)) < 2 or len(Tva) < 2:
                continue
            rf.fit(Ftr, Ttr)
            pred = rf.predict(Fva)
            try:
                cv_scores.append(r2_score(Tva, pred))
            except Exception:
                pass
        mean_cv = np.mean(cv_scores) if len(cv_scores)>0 else -1e9
        if mean_cv > best_cv:
            best_cv = mean_cv; best_model=rf; best_conf=conf
else:
    # fallback estável
    best_conf = RF_GRID[1]
    best_model = RandomForestRegressor(**best_conf)

rf_temp = best_model.fit(F_tr, T_tr)

# prints RF-Temp
train_r2 = float(rf_temp.score(F_tr, T_tr)) if len(np.unique(T_tr))>1 else np.nan
print("\n== RF-Temp (features globais) ==")
if np.isfinite(train_r2):
    if best_cv>-1e8:
        print(f"R²(treino) = {train_r2:.3f} | R²({('K-fold')}) ≈ {best_cv:.3f}")
    else:
        print(f"R²(treino) = {train_r2:.3f}")
else:
    print("R²(treino) indisponível (variação insuficiente).")
print("Importâncias das features:")
for name, imp in sorted(zip(feat_names, rf_temp.feature_importances_), key=lambda x:-x[1]):
    print(f"{name:>12s}: {imp:.3f}")

# modelos feature ~ T (para alvos a 20°C)
feat_models = fit_feature_vs_temp_models(F_tr, T_tr, feat_names)
feat_targets_20 = feature_targets_at_ref(feat_models, REF_TEMP)
print("\nTargets de features em 20°C (via regressão no treino):")
for k,v in feat_targets_20.items():
    print(f"{k:>12s}: {v:.6f}")

# ===================== COMPENSAÇÃO POR VARIÁVEIS (RF) =====================
caps = dict(gain_frac=CAP_GAIN_FRAC, offset_frac=CAP_OFFSET_FRAC, tilt_frac=CAP_TILT_FRAC)
t0 = time.time()
Y_te_hat, _targets = compensate_set_by_features(X_te, fhz, feat_models, REF_TEMP, y_ref, caps, smooth_win=SMOOTH_WIN)
print(f"\n[INFO] Compensação por features aplicada em {time.time()-t0:.2f}s")

# ===================== MÉTRICAS PADRÃO (sem overlay) =====================
Y_ref_te = np.tile(y_ref, (X_te.shape[0],1))
m_rf_ref  = eval_all_metrics(Y_ref_te, Y_te_hat); print_block("FINAL vs REF",  m_rf_ref)
m_rf_orig = eval_all_metrics(X_te,     Y_te_hat); print_block("FINAL vs ORIG", m_rf_orig)

# ===== Blocos RAW/ALIGNED (Grupo 1 vs REF e Grupo 2 vs ORIG), com overlay
def print_group_blocks(title_prefix, f, REF, CUR, align_against="REF"):
    # RAW
    g_raw = group_metrics(f, REF, CUR)
    print(f"\n### {title_prefix} — RAW ###")
    for k in ["Corr","SAM_deg","CCDM","RMSD","mean_diff","amp_diff","centroid_diff_hz","slope_diff"]:
        print(f"{k:>16s}: {g_raw[k]:10.4f}")
    # ALIGNED (apenas para avaliar forma, removendo lag/gain/offset globais)
    lag, a, b, aligned = best_affine_overlay(f, REF, CUR)
    print(f"\n### {title_prefix} — ALIGNED ###")
    print(f"{'lag_bins':>16s}: {lag:10.4f}")
    print(f"{'gain':>16s}: {a:10.4f}")
    print(f"{'offset':>16s}: {b:10.4f}")
    g_al = group_metrics(f, REF, aligned)
    print(f"\n### {title_prefix} — ALIGNED (métricas ALIGNED) ###")
    for k in ["Corr","SAM_deg","CCDM","RMSD","mean_diff","amp_diff","centroid_diff_hz","slope_diff"]:
        print(f"{k:>16s}: {g_al[k]:10.4f}")

# Grupo 1: sensíveis à temperatura => comparar Y_hat vs REF
print_group_blocks("Grupo 1 (vs REF) — RF", fhz, y_ref, Y_te_hat[0])  # exemplo com a primeira amostra
# Grupo 2: estrutura que deve se manter => comparar Y_hat vs ORIG
print_group_blocks("Grupo 2 (vs ORIG) — RF", fhz, X_te[0], Y_te_hat[0])

# Índices estilo Dias (médias dos blocos RAW)
print("\n### Índices do Dias — RAW")
print(f"(vs REF)  RMSD={m_rf_ref['RMSD']:.4f} | CCDM={m_rf_ref['CCDM']:.4f}")
print(f"(vs ORIG) RMSD={m_rf_orig['RMSD']:.4f} | CCDM={m_rf_orig['CCDM']:.4f}")

# ===================== PARK (comparação) =====================
print("\n[COMPARAÇÃO] Park clássico (fora do RF)...")
Y_te_park, park_lags, park_deltas = park_batch(X_te, y_ref)
m_pk_ref  = eval_all_metrics(Y_ref_te, Y_te_park); print_block("PARK vs REF",  m_pk_ref)
m_pk_orig = eval_all_metrics(X_te,     Y_te_park); print_block("PARK vs ORIG", m_pk_orig)

# versões RAW/ALIGNED do Park (na amostra 0 para ilustrar)
print_group_blocks("Grupo PARK (vs REF)",  fhz, y_ref,  Y_te_park[0])
print_group_blocks("Grupo PARK (vs ORIG)", fhz, X_te[0], Y_te_park[0])

print("\n### Índices do Dias — RAW (PARK)")
print(f"(vs REF)  RMSD={m_pk_ref['RMSD']:.4f} | CCDM={m_pk_ref['CCDM']:.4f}")
print(f"(vs ORIG) RMSD={m_pk_orig['RMSD']:.4f} | CCDM={m_pk_orig['CCDM']:.4f}")

# ===================== Checagem RF-Temp nas curvas finais =====================
F_final, _ = compute_features(Y_te_hat, fhz)
T_hat_final = rf_temp.predict(F_final)
print("\n### Checagem RF-Temp nas curvas finais ###")
print(f"média={float(np.mean(T_hat_final)):.2f}°C | desvio={float(np.std(T_hat_final)):.2f}°C | MAE vs {REF_TEMP}°C={float(np.mean(np.abs(T_hat_final-REF_TEMP))):.2f}°C")

# ===================== PLOTS =====================
def _prep_plot():
    plt.rcParams.update({
        "figure.figsize": (9.2, 5.0),
        "axes.grid": True, "grid.alpha": 0.28,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelsize": 12, "axes.titlesize": 13,
        "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 10, "lines.linewidth": 1.8,
    })

def plot_rf(i=0, save=False, prefix="rf_comp"):
    _prep_plot()
    fhz_khz = fkHz
    T_real = float(te_restr.iloc[i]["temp_c"])
    # “temperatura percebida” pelo RF-Temp, antes/depois
    F_orig, _ = compute_features(X_te[i][None,:], fhz)
    F_comp, _ = compute_features(Y_te_hat[i][None,:], fhz)
    T_pred_orig  = float(rf_temp.predict(F_orig)[0])
    T_pred_final = float(rf_temp.predict(F_comp)[0])

    fig, ax = plt.subplots()
    ax.plot(fhz_khz, X_te[i],     label=f"Original @ {T_real:.0f} °C (RF≈{T_pred_orig:.1f} °C)", color="#1f77b4")
    ax.plot(fhz_khz, y_ref,       label=f"Referência @ {REF_TEMP} °C", color="#ff7f0e")
    ax.plot(fhz_khz, Y_te_hat[i], label=f"Compensada (RF≈{T_pred_final:.1f} °C)", color="#2ca02c")
    ax.set_title(f"Amostra {i} — Compensação por variáveis (mean/amp/slope + micro-shift + âncora)")
    ax.set_xlabel("Frequência (kHz)"); ax.set_ylabel("Re{Z}")
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    if save: fig.savefig(f"{prefix}_i{i}.png", dpi=300)
    plt.show()

def plot_park(i=0, save=False, prefix="park_comp"):
    _prep_plot()
    fhz_khz = fkHz
    T_real = float(te_restr.iloc[i]["temp_c"])
    fig, ax = plt.subplots()
    ax.plot(fhz_khz, X_te[i],       label=f"Original @ {T_real:.0f} °C",  color="#1f77b4")
    ax.plot(fhz_khz, y_ref,         label=f"Referência @ {REF_TEMP} °C", color="#ff7f0e")
    ax.plot(fhz_khz, Y_te_park[i],  label=f"Park (comp.)",               color="#9467bd")
    ax.set_title(f"Amostra {i} — Park (1999) – comparação")
    ax.set_xlabel("Frequência (kHz)"); ax.set_ylabel("Re{Z}")
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    if save: fig.savefig(f"{prefix}_i{i}.png", dpi=300)
    plt.show()

# Exemplos de plot:
plot_rf(i=0, save=False)
plot_park(i=0, save=False)


In [ ]:
plot_rf(i=0, save=False)
plot_park(i=0, save=False)

In [ ]:
def thermal_score_weighted_rel(y_ref, y_hat, f, rf_model, feat_names, eps=1e-9):
    """Score térmico robusto: erro relativo ponderado pelas importâncias do RF."""
    feats_ref, _ = compute_features(y_ref[None,:], f)
    feats_hat, _ = compute_features(y_hat[None,:], f)

    importances = rf_model.feature_importances_
    weights = importances / (np.sum(importances) + eps)

    diffs_rel = np.abs(feats_ref[0] - feats_hat[0]) / (np.abs(feats_ref[0]) + eps)
    E_therm = np.sum(weights * diffs_rel)

    return 1.0 / (1.0 + E_therm)


def structural_score_shape(y_orig, y_hat, eps=1e-12):
    """Score estrutural: correlação de forma (z-score)."""
    def norm(x): return (x - np.mean(x)) / (np.std(x) + eps)
    yo, yh = norm(y_orig), norm(y_hat)
    num = np.dot(yo, yh)
    den = np.linalg.norm(yo) * np.linalg.norm(yh) + eps
    return float(num / den)


def consistency_index_robust(y_ref, y_orig, y_hat, f, rf_model, feat_names, eps=1e-12):
    """Índice robusto: ST com erros relativos + combinação harmônica com SE."""
    st = thermal_score_weighted_rel(y_ref, y_hat, f, rf_model, feat_names)
    se = structural_score_shape(y_orig, y_hat)
    se_pos = (se + 1) / 2.0  # reescala [-1,1] → [0,1]

    # média harmônica robusta
    return (2 * st * se_pos) / (st + se_pos + eps)


In [ ]:
print("\n### Consistência Robusta (ICB_robusto) — RF")
icb_rf = [consistency_index_robust(y_ref, X_te[i], Y_te_hat[i], fhz, rf_temp, feat_names)
          for i in range(X_te.shape[0])]
for i,val in enumerate(icb_rf):
    print(f"Amostra {i}: ICB_robusto={val:.4f}")
print(f"ICB_robusto RF (médio) = {np.mean(icb_rf):.4f} ± {np.std(icb_rf):.4f}")

print("\n### Consistência Robusta (ICB_robusto) — Park")
icb_pk = [consistency_index_robust(y_ref, X_te[i], Y_te_park[i], fhz, rf_temp, feat_names)
          for i in range(X_te.shape[0])]
for i,val in enumerate(icb_pk):
    print(f"Amostra {i}: ICB_robusto={val:.4f}")
print(f"ICB_robusto Park (médio) = {np.mean(icb_pk):.4f} ± {np.std(icb_pk):.4f}")


In [ ]:
def ST_puro(y_ref, y_hat, f, rf_model, feat_names, eps=1e-9):
    """Score térmico puro: só mean, amp, slope, centroid."""
    feats_ref, _ = compute_features(y_ref[None,:], f)
    feats_hat, _ = compute_features(y_hat[None,:], f)

    # filtra apenas as 4 features
    sel = ["mean", "amp", "slope", "centroid"]
    importances = dict(zip(feat_names, rf_model.feature_importances_))
    weights = np.array([importances[n] for n in sel], float)
    weights = weights / (np.sum(weights) + eps)

    # pega valores correspondentes
    idxs = [feat_names.index(n) for n in sel]
    fr, fh = feats_ref[0, idxs], feats_hat[0, idxs]

    diffs_rel = np.abs(fr - fh) / (np.abs(fr) + eps)
    E_therm = np.sum(weights * diffs_rel)

    return 1.0 / (1.0 + E_therm)


def SE_puro(y_orig, y_hat, win=11, eps=1e-12):
    """Score estrutural puro: correlação de forma suavizada."""
    def smooth(x, w):
        if w <= 1: return x
        return np.convolve(x, np.ones(w)/w, mode="same")
    def norm(x): return (x - np.mean(x)) / (np.std(x) + eps)

    yo, yh = smooth(y_orig, win), smooth(y_hat, win)
    yo, yh = norm(yo), norm(yh)

    num = np.dot(yo, yh)
    den = np.linalg.norm(yo) * np.linalg.norm(yh) + eps
    return float((num/den + 1) / 2.0)  # já reescalado [0,1]


def IFF_index(y_ref, y_orig, y_hat, f, rf_model, feat_names, win=11, eps=1e-12):
    """Índice Final de Fidelidade (IFF) = combinação de ST_puro e SE_puro."""
    st = ST_puro(y_ref, y_hat, f, rf_model, feat_names)
    se = SE_puro(y_orig, y_hat, win)
    return (2 * st * se) / (st + se + eps)


In [ ]:
print("\n### IFF — RF")
iff_rf = [IFF_index(y_ref, X_te[i], Y_te_hat[i], fhz, rf_temp, feat_names)
          for i in range(X_te.shape[0])]
for i,val in enumerate(iff_rf):
    print(f"Amostra {i}: IFF={val:.4f}")
print(f"IFF RF (médio) = {np.mean(iff_rf):.4f} ± {np.std(iff_rf):.4f}")

print("\n### IFF — Park")
iff_pk = [IFF_index(y_ref, X_te[i], Y_te_park[i], fhz, rf_temp, feat_names)
          for i in range(X_te.shape[0])]
for i,val in enumerate(iff_pk):
    print(f"Amostra {i}: IFF={val:.4f}")
print(f"IFF Park (médio) = {np.mean(iff_pk):.4f} ± {np.std(iff_pk):.4f}")


In [ ]:
# Listas para salvar os valores
st_rf, se_rf = [], []
st_pk, se_pk = [], []

for i in range(X_te.shape[0]):
    st_rf.append(ST_puro(y_ref, Y_te_hat[i], fhz, rf_temp, feat_names))
    se_rf.append(SE_puro(X_te[i], Y_te_hat[i]))

    st_pk.append(ST_puro(y_ref, Y_te_park[i], fhz, rf_temp, feat_names))
    se_pk.append(SE_puro(X_te[i], Y_te_park[i]))

# Converter para arrays
st_rf, se_rf = np.array(st_rf), np.array(se_rf)
st_pk, se_pk = np.array(st_pk), np.array(se_pk)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,6))
plt.scatter(st_rf, se_rf, c="green", label="RF", s=70, marker="o", alpha=0.8)
plt.scatter(st_pk, se_pk, c="purple", label="Park", s=70, marker="^", alpha=0.8)

plt.axhline(0.5, color="gray", linestyle="--", lw=1)
plt.axvline(0.5, color="gray", linestyle="--", lw=1)

plt.xlabel("ST_puro (atributos térmicos)", fontsize=12)
plt.ylabel("SE_puro (forma estrutural)", fontsize=12)
plt.title("Separação: Térmico vs Estrutural", fontsize=13)
plt.legend(frameon=False)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
print("\n### ST_puro e SE_puro — RF")
for i in range(X_te.shape[0]):
    st_val = ST_puro(y_ref, Y_te_hat[i], fhz, rf_temp, feat_names)
    se_val = SE_puro(X_te[i], Y_te_hat[i])
    print(f"Amostra {i}: ST_puro={st_val:.4f} | SE_puro={se_val:.4f}")

print("\n### ST_puro e SE_puro — Park")
for i in range(X_te.shape[0]):
    st_val = ST_puro(y_ref, Y_te_park[i], fhz, rf_temp, feat_names)
    se_val = SE_puro(X_te[i], Y_te_park[i])
    print(f"Amostra {i}: ST_puro={st_val:.4f} | SE_puro={se_val:.4f}")
